<a href="https://colab.research.google.com/github/AllaYermilko/DTA-2026/blob/main/ML/logreg_pipeline_TASKS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Воркбук: логістична регресія + Pipeline

Просте тренування на дві теми:
- **Логістична регресія** - класика класифікації, що дає ймовірності й інтерпретовні коефіцієнти.
- **Pipeline** - складаємо препроцесинг (масштабування + кодування) і модель в один надійний конвеєр.

**Набір даних:** клієнти сервісу (`clients`). Ціль - `upgraded` (1 = перейшов на преміум, 0 = ні).

| Стовпець | Що це | Тип |
|---|---|---|
| `age` | вік | число |
| `tenure` | місяців із сервісом | число |
| `usage` | годин/міс використання | число |
| `support` | звернень у підтримку | число |
| `plan` | тариф (базовий/стандарт/сімейний) | категорія |
| `region` | регіон | категорія |
| `upgraded` | перейшов на преміум - **ціль** | 0/1 |

**Як працювати:** запусти «Підготовку даних», іди по кроках, заповнюй `# TODO`. Підказки - під кожним кроком.


---

## 🔧 Підготовка даних (просто запусти)

In [1]:
# ▶️ Просто запусти цю комірку — вона готує дані. Міняти нічого не треба.
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)

# Задача: чи перейде клієнт на ПРЕМІУМ-підписку (1 = так, 0 = ні)
N = 900
age          = np.random.randint(18, 70, N)                       # вік
tenure       = np.random.randint(1, 60, N)                        # місяців із сервісом
usage        = np.random.normal(80, 35, N).clip(0, 220).round(0)  # годин/міс використання
support      = np.random.poisson(1.3, N)                          # звернень у підтримку

plan   = np.random.choice(["базовий", "стандарт", "сімейний"], N, p=[.45, .35, .20])
plan_bonus = pd.Series({"базовий": -0.4, "стандарт": 0.3, "сімейний": 1.1})

region = np.random.choice(["північ", "південь", "схід", "захід"], N)
region_bonus = pd.Series({"північ": 0.1, "південь": -0.1, "схід": 0.0, "захід": 0.2})

logit = (0.03*usage + 0.045*tenure - 0.35*support - 0.012*age
         + plan_bonus[plan].values + region_bonus[region].values
         - 3.0 + np.random.normal(0, 0.8, N))
upgraded = (logit > 0).astype(int)

clients = pd.DataFrame({
    "age": age, "tenure": tenure, "usage": usage.astype(int), "support": support,
    "plan": plan, "region": region, "upgraded": upgraded,
})

print("✅ Дані готові. Таблиця clients:", clients.shape)
print("Частка тих, хто перейшов на преміум:", f"{clients['upgraded'].mean():.0%}")

✅ Дані готові. Таблиця clients: (900, 7)
Частка тих, хто перейшов на преміум: 48%


In [2]:
# Подивись на дані
clients.head()

,age,tenure,usage,support,plan,region,upgraded
0,56,17,79,4,базовий,схід,0
1,69,5,61,2,базовий,південь,0
2,46,29,24,0,базовий,північ,0
3,32,4,100,0,стандарт,захід,1
4,60,10,52,0,стандарт,захід,0


---
### Крок 1. Розвідка: баланс класів і типи ознак
Виведи частку кожного класу в `upgraded` і визнач, які стовпці числові, а які категорійні.

*Підказка:* `clients["upgraded"].value_counts(normalize=True)`.

In [6]:
# TODO: виведи баланс класів
print("--- Баланс класів ---")
print(clients["upgraded"].value_counts(normalize=True))

# 2. Перевіряємо типи даних
print("\n--- Типи ознак ---")
print(clients.dtypes)

--- Баланс класів ---
upgraded
0    0.515556
1    0.484444
Name: proportion, dtype: float64

--- Типи ознак ---
age          int64
tenure       int64
usage        int64
support      int64
plan        object
region      object
upgraded     int64
dtype: object


✍️ Випиши списки стовпців (знадобляться далі):
> числові: age, tenure, usage, support  
> категорійні: plan, region

### Крок 2. X, y і поділ train / test
- `X` — усі стовпці, КРІМ `upgraded`. `y` — `upgraded`.
- Поділ: 20% у тест, `random_state=RANDOM_STATE`, **`stratify=y`** (щоб пропорція класів збереглась).

*Підказка:* `train_test_split(X, y, test_size=.., random_state=.., stratify=..)`.

In [8]:
from sklearn.model_selection import train_test_split

# TODO: створи X, y та зроби поділ
X = clients[['age', 'tenure', 'usage', 'support', 'plan', 'region']]
y = clients['upgraded']

# Робимо поділ на навчання (train) та тест (test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Розмір навчальної вибірки X_train:", X_train.shape)
print("Розмір тестової вибірки X_test:", X_test.shape)

Розмір навчальної вибірки X_train: (720, 6)
Розмір тестової вибірки X_test: (180, 6)


### Крок 3. Опиши, що робити з кожним типом стовпців (`ColumnTransformer`)
Числові — **масштабувати** (`StandardScaler`); категорійні — **One-Hot** (`OneHotEncoder`).
Логістичній регресії масштабування потрібне (у ній є регуляризація).

*Підказка:*
```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = [..]
cat_cols = [..]

preprocess = ColumnTransformer([
    ("num", .., num_cols),
    ("cat", .., cat_cols),
])
```

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# TODO: задай num_cols, cat_cols і збери preprocess

num_cols = ['age', 'tenure', 'usage', 'support']
cat_cols = ['plan', 'region']

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])


### Крок 4. Збери повний `Pipeline`: препроцесинг + модель
Поклади `preprocess` і `LogisticRegression(max_iter=1000)` в один `Pipeline`.

*Підказка:*
```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("prep", ..),
    ("model", ..),
])
```

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# TODO: збери pipe
pipe = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

### Крок 5. Навчи конвеєр і виміряй accuracy на тесті
Один виклик `.fit(X_train, y_train)` прожене дані через усі кроки.

*Підказка:* `pipe.fit(...)`, далі `pipe.score(X_test, y_test)`.

In [15]:
# TODO: навчи pipe і виведи accuracy на тесті
# 1. Навчаємо конвеєр (він сам зробить препроцесинг і навчить логістичну регресію)
pipe.fit(X_train, y_train)

# 2. Рахуємо та виводимо точність моделі на тестових даних
accuracy_test = pipe.score(X_test, y_test)
print('Accuracy на тесті:', round(accuracy_test, 3))

Accuracy на тесті: 0.844


### Крок 6. Деталізована оцінка: матриця плутанини й звіт
Передбач класи на тесті, побудуй `confusion_matrix` і `classification_report`.

*Підказка:* `pipe.predict(X_test)`; `confusion_matrix(...)`; `classification_report(...)`.

In [16]:
from sklearn.metrics import confusion_matrix, classification_report

# TODO: передбач, виведи матрицю плутанини та звіт
# 1. Отримуємо передбачення класів (0 або 1) для тестової вибірки
y_pred = pipe.predict(X_test)

# 2. Будуємо та виводимо матрицю плутанини
print("--- Матриця плутанини (Confusion Matrix) ---")
print(confusion_matrix(y_test, y_pred))
print()

# 3. Генеруємо та виводимо детальний звіт про метрики якості
print("--- Звіт про класифікацію (Classification Report) ---")
print(classification_report(y_test, y_pred))

--- Матриця плутанини (Confusion Matrix) ---
[[80 13]
 [15 72]]

--- Звіт про класифікацію (Classification Report) ---
              precision    recall  f1-score   support

           0       0.84      0.86      0.85        93
           1       0.85      0.83      0.84        87

    accuracy                           0.84       180
   macro avg       0.84      0.84      0.84       180
weighted avg       0.84      0.84      0.84       180



### Крок 7. Ймовірності + ROC-AUC
Логістична регресія дає не лише мітку, а й **ймовірність**. Дістань ймовірність класу «1» і порахуй ROC-AUC.

*Підказка:* `proba = pipe.predict_proba(X_test)[:, 1]`; `roc_auc_score(y_test, proba)`.

In [18]:
from sklearn.metrics import roc_auc_score

# TODO: дістань proba та порахуй ROC-AUC
# 1. Дістаємо ймовірність того, що клієнт перейде на преміум (клас 1)
proba = pipe.predict_proba(X_test)[:, 1]

# 2. Рахуємо та виводимо метрику ROC-AUC на екран
roc_auc = roc_auc_score(y_test, proba)
print("ROC-AUC score на тесті:", round(roc_auc, 3))

ROC-AUC score на тесті: 0.927


### Крок 8. 🔑 Інтерпретація коефіцієнтів
Дістань назви ознак після препроцесингу й коефіцієнти моделі. Знак: **+ підвищує** ймовірність переходу, − знижує.

*Підказка:*
```python
names = ..
coefs = ..
```
Зведи у `DataFrame` і відсортуй за модулем.

In [20]:
# TODO: побудуй таблицю "ознака — коефіцієнт", відсортовану за |коеф.|

feat_names = pipe.named_steps['prep'].get_feature_names_out()
coefs = pipe.named_steps['model'].coef_[0]
coef_df = pd.DataFrame({
    'features': feat_names,
    'coef': coefs
}).sort_values('coef', key=abs, ascending=False)

print('Вплив ознак на перехід на ПРЕМІУМ (+ підвищує ймовірність, - знижує):')
coef_df



Вплив ознак на перехід на ПРЕМІУМ (+ підвищує ймовірність, - знижує):


,features,coef
2,num__usage,2.027135
1,num__tenure,1.648386
6,cat__plan_сімейний,1.411953
4,cat__plan_базовий,-1.339728
3,num__support,-0.835947
7,cat__region_захід,0.537637
0,num__age,-0.406875
10,cat__region_схід,-0.340620
8,cat__region_південь,-0.189350
9,cat__region_північ,0.136239


✍️ **Відповідь словами:**
> Найсильніше підвищує шанс переходу на преміум num__usage	(2.027135). У цієї ознаки найбільший позитивний коефіцієнт (+2.02). Це означає: що більше годин на місяць людина проводить у сервісі, то вища ймовірність, що вона захоче купити преміум.  
Знижує шанс переходу - cat__plan_базовий	-1.339728. У цієї ознаки найбільше за модулем від'ємне значення (-1.33). Це означає, що люди на базовому тарифі найменше схильні витрачати гроші на оновлення підписки.

### Крок 9. Прогноз для нового клієнта
Конвеєр приймає **сирі** дані — кодувати/масштабувати вручну не треба. Створи клієнта й виведи і рішення, і ймовірність.

Клієнт: вік 30, tenure 24, usage 120, support 0, plan «сімейний», region «захід».

*Підказка:* `pd.DataFrame([{...}])` з тими самими назвами стовпців → `pipe.predict_proba(...)[0, 1]`.

In [21]:
# TODO: створи new_client, виведи рішення та ймовірність переходу
new_client = pd.DataFrame([{
    'age': 30,
    'tenure': 24,
    'usage': 120,
    'support': 0,
    'plan': 'сімейний',
    'region': 'захід'
}])

display(new_client)

print('\nКлієнт -', 'ПІДЕ' if pipe.predict(new_client) else 'Залишиться')

,age,tenure,usage,support,plan,region
0,30,24,120,0,сімейний,захід



Клієнт - ПІДЕ


### Крок 10. Чесна оцінка: крос-валідація всього конвеєра
Прожени `pipe` через `cross_val_score` (cv=5, scoring="roc_auc"). Бо весь препроцесинг усередині Pipeline — кожен фолд обробляється окремо, **без витоку**.

*Підказка:* `cross_val_score(pipe, X, y, cv=5, scoring="roc_auc")`.

In [22]:
from sklearn.model_selection import cross_val_score

# TODO: крос-валідація, виведи середнє ± розкид
cv = cross_val_score(pipe, X, y, cv=5, scoring='roc_auc')
print(f'CV accuracy: {np.round(cv, 3)} | середнє: {cv.mean():.3f} ± {cv.std():.3}')

CV accuracy: [0.934 0.931 0.94  0.891 0.921] | середнє: 0.923 ± 0.0175


---
# ⭐ Бонус (необов'язково)
1. **Навіщо масштабування?** Збери другий конвеєр **без** `StandardScaler` (числові — `passthrough`) і порівняй ROC-AUC. Сильно змінилось?
```python
("num", "passthrough", num_cols)
```
2. **Дисбаланс класів.** Додай у `LogisticRegression(class_weight="balanced")` і подивись, як зміняться recall для класу «1» та матриця плутанини.
```python
LogisticRegression(max_iter=1000, class_weight="balanced")
```
3. **Поріг рішення.** Замість порогу 0.5 спробуй 0.3 (`proba >= 0.3`). Як зміняться precision і recall?
```python
# 3. Поріг 0.3 замість 0.5
import numpy as np
proba = pipe.predict_proba(X_test)[:, 1]
for thr in [0.5, 0.3]:
    pred_thr = (proba >= thr).astype(int)
    cm = confusion_matrix(y_test, pred_thr)
    print(f"Поріг {thr}: матриця\n{cm}")
print("→ нижчий поріг ловить більше 'так' (вищий recall), але росте й хибних тривог (нижчий precision)")
```

In [ ]:
# Місце для бонусних експериментів

---
# 🧠 Питання на розуміння (без коду)
1. Чому логістична регресія — це **класифікація**, попри слово «регресія» в назві?  
- Модель називається «регресією», бо всередині вона рахує відсотки ймовірності (числа від 0 до 100). Але вважається «класифікацією», тому що її фінальна відповідь — це завжди чіткий вибір між двома класами  

2. Що показує `predict_proba` і чим воно корисніше за `predict` для бізнесу?  
- predict просто ділить клієнтів на два табори (1 або 0), а predict_proba показує точну ймовірність події у відсотках. Для бізнесу це корисніше, бо дозволяє сортувати клієнтів від «найгарячіших» до «холодних», ефективно витрачати маркетинговий бюджет і гнучко керувати ризиками компанії.  

3. Навіщо взагалі загортати кроки в `Pipeline` — що поганого станеться, якщо масштабувати дані **до** `train_test_split`?  
- Масштабування даних до поділу призводить до витоку даних (data leakage), оскільки модель заздалегідь дізнається інформацію про середні значення тестової вибірки. Pipeline потрібен для того, щоб суворо ізолювати тестові дані від тренувальних, автоматично виконувати препроцесинг у правильному порядку та захистити код від помилок  

4. Логістичній регресії масштабування потрібне, а дереву рішень — ні. Чому?  
- Логістична регресія єметричним алгоритмом, який об'єднує всі ознаки в одне математичне рівняння. Якщо дані не масштабувати, великі числа просто "заглушать" малі. Дерево рішень працює шляхом послідовного розділення ознак за логічними правилами "Більше/Менше" окремо для кожного стовпця, тому масштаб та розмір чисел взагалі не впливають на його роботу  

5. Коефіцієнт `support` від'ємний. Як прочитати це вголос для керівника?  
- Кожне додаткове звернення клієнта в підтримку знижує його інтерес до преміум-підписки. Що частіше людина стикається з проблемами, то менша ймовірність, що вона принесе нам гроші

> 🎯 Якщо зібрав робочий Pipeline і впевнено читаєш коефіцієнти — ти володієш найбільш «продакшн-готовим» патерном класичного ML.